In [33]:
# Cell 1: Import và cấu hình
from pathlib import Path

import pandas as pd
from IPython.display import display


# Thư mục gốc của project
PROJECT_ROOT = Path(r"D:\OLAP")

# Thư mục chứa dữ liệu Raw
RAW_DIR = PROJECT_ROOT / "Data" / "Raw"

# Thư mục lưu kết quả profiling
PROFILING_DIR = PROJECT_ROOT / "Data" / "Profiling"


# Đường dẫn ba file Raw
RAW_FILES = {
    "Collisions": RAW_DIR / "collisions_2021_2025.csv",
    "Vehicles": RAW_DIR / "vehicles_2021_2025.csv",
    "Casualties": RAW_DIR / "casualties_2021_2025.csv",
}


# Tạo thư mục Profiling nếu chưa tồn tại
PROFILING_DIR.mkdir(parents=True, exist_ok=True)

print("Đã khai báo cấu hình.")

Đã khai báo cấu hình.


In [34]:
# Cell 2: Danh sách cột cần profiling
SELECTED_COLUMNS = {
    "Collisions": [
        "collision_index",
        "date",
        "time",
        "longitude",
        "latitude",
        "local_authority_ons_district",
        "collision_severity",
        "first_road_class",
        "road_type",
        "speed_limit",
        "junction_detail",
        "junction_control",
        "pedestrian_crossing",
        "light_conditions",
        "weather_conditions",
        "road_surface_conditions",
        "urban_or_rural_area",
    ],

    "Vehicles": [
        "collision_index",
        "vehicle_reference",
        "vehicle_type",
        "vehicle_manoeuvre",
        "age_band_of_driver",
        "sex_of_driver",
        "skidding_and_overturning",
        "vehicle_leaving_carriageway",
    ],

    "Casualties": [
        "collision_index",
        "vehicle_reference",
        "casualty_reference",
        "casualty_class",
        "sex_of_casualty",
        "age_band_of_casualty",
        "casualty_severity",
        "casualty_type",
    ],
}


NATURAL_KEYS = {
    "Collisions": [
        "collision_index",
    ],

    "Vehicles": [
        "collision_index",
        "vehicle_reference",
    ],

    "Casualties": [
        "collision_index",
        "casualty_reference",
    ],
}


for table_name, columns in SELECTED_COLUMNS.items():
    print(f"{table_name}: {len(columns)} cột")


Collisions: 17 cột
Vehicles: 8 cột
Casualties: 8 cột


In [35]:
# Cell 3: Kiểm tra đường dẫn
file_check_results = []

for table_name, file_path in RAW_FILES.items():
    file_exists = file_path.exists()

    file_size_mb = (
        file_path.stat().st_size / (1024 ** 2)
        if file_exists
        else None
    )

    file_check_results.append(
        {
            "Table": table_name,
            "File": file_path.name,
            "Exists": file_exists,
            "Size_MB": (
                round(file_size_mb, 2)
                if file_size_mb is not None
                else None
            ),
            "Path": str(file_path),
        }
    )

file_check_df = pd.DataFrame(file_check_results)

display(file_check_df)

,Table,File,Exists,Size_MB,Path
0,Collisions,collisions_2021_2025.csv,True,93.14,D:\OLAP\Data\Raw\collisions_2021_2025.csv
1,Vehicles,vehicles_2021_2025.csv,True,98.94,D:\OLAP\Data\Raw\vehicles_2021_2025.csv
2,Casualties,casualties_2021_2025.csv,True,49.95,D:\OLAP\Data\Raw\casualties_2021_2025.csv


In [36]:
# Cell 4: Kiểm tra header
EXPECTED_TOTAL_COLUMNS = {
    "Collisions": 44,
    "Vehicles": 32,
    "Casualties": 23,
}


header_check_results = []

for table_name, file_path in RAW_FILES.items():

    # nrows=0 nghĩa là chỉ đọc tên cột, không đọc dữ liệu
    header = pd.read_csv(
        file_path,
        nrows=0,
        encoding="utf-8-sig",
    )

    actual_columns = header.columns.tolist()
    required_columns = SELECTED_COLUMNS[table_name]

    missing_columns = sorted(
        set(required_columns) - set(actual_columns)
    )

    header_check_results.append(
        {
            "Table": table_name,
            "Expected_Total_Columns": EXPECTED_TOTAL_COLUMNS[table_name],
            "Actual_Total_Columns": len(actual_columns),
            "Required_Columns": len(required_columns),
            "Missing_Required_Columns": (
                ", ".join(missing_columns)
                if missing_columns
                else "None"
            ),
            "Header_Status": (
                "PASS"
                if (
                    len(actual_columns)
                    == EXPECTED_TOTAL_COLUMNS[table_name]
                    and not missing_columns
                )
                else "CHECK"
            ),
        }
    )


header_check_df = pd.DataFrame(header_check_results)

display(header_check_df)

,Table,Expected_Total_Columns,Actual_Total_Columns,Required_Columns,Missing_Required_Columns,Header_Status
0,Collisions,44,44,17,None,PASS
1,Vehicles,32,32,8,None,PASS
2,Casualties,23,23,8,None,PASS


In [37]:
# Cell 5 Khai báo kiểu dữ liệu Raw
RAW_DTYPES = {
    "Collisions": {
        "collision_index": "string",
        "date": "string",
        "time": "string",
        "longitude": "float64",
        "latitude": "float64",
        "local_authority_ons_district": "category",
        "collision_severity": "category",
        "first_road_class": "category",
        "road_type": "category",
        "speed_limit": "category",
        "junction_detail": "category",
        "junction_control": "category",
        "pedestrian_crossing": "category",
        "light_conditions": "category",
        "weather_conditions": "category",
        "road_surface_conditions": "category",
        "urban_or_rural_area": "category",
    },

    "Vehicles": {
        "collision_index": "string",
        "vehicle_reference": "Int64",
        "vehicle_type": "category",
        "vehicle_manoeuvre": "category",
        "age_band_of_driver": "category",
        "sex_of_driver": "category",
        "skidding_and_overturning": "category",
        "vehicle_leaving_carriageway": "category",
    },

    "Casualties": {
        "collision_index": "string",
        "vehicle_reference": "Int64",
        "casualty_reference": "Int64",
        "casualty_class": "category",
        "sex_of_casualty": "category",
        "age_band_of_casualty": "category",
        "casualty_severity": "category",
        "casualty_type": "category",
    },
}

print("Đã khai báo kiểu dữ liệu Raw.")

Đã khai báo kiểu dữ liệu Raw.


In [38]:
# Cell 6 Đọc 33 cột Raw
print("Đang đọc Collisions...")

df_collisions = pd.read_csv(
    RAW_FILES["Collisions"],
    usecols=SELECTED_COLUMNS["Collisions"],
    dtype=RAW_DTYPES["Collisions"],
    encoding="utf-8-sig",
    low_memory=False,
)


print("Đang đọc Vehicles...")

df_vehicles = pd.read_csv(
    RAW_FILES["Vehicles"],
    usecols=SELECTED_COLUMNS["Vehicles"],
    dtype=RAW_DTYPES["Vehicles"],
    encoding="utf-8-sig",
    low_memory=False,
)


print("Đang đọc Casualties...")

df_casualties = pd.read_csv(
    RAW_FILES["Casualties"],
    usecols=SELECTED_COLUMNS["Casualties"],
    dtype=RAW_DTYPES["Casualties"],
    encoding="utf-8-sig",
    low_memory=False,
)


print("Đã đọc xong ba file.")

Đang đọc Collisions...
Đang đọc Vehicles...
Đang đọc Casualties...
Đã đọc xong ba file.


In [39]:
# Cell 7 Kiểm tra số dòng, số cột và RAM
EXPECTED_ROWS = {
    "Collisions": 513_801,
    "Vehicles": 937_265,
    "Casualties": 652_821,
}


DATAFRAMES = {
    "Collisions": df_collisions,
    "Vehicles": df_vehicles,
    "Casualties": df_casualties,
}


shape_results = []

for table_name, dataframe in DATAFRAMES.items():

    actual_rows = len(dataframe)
    actual_columns = len(dataframe.columns)

    memory_mb = (
        dataframe.memory_usage(deep=True).sum()
        / (1024 ** 2)
    )

    shape_results.append(
        {
            "Table": table_name,
            "Expected_Rows": EXPECTED_ROWS[table_name],
            "Actual_Rows": actual_rows,
            "Rows_Match": (
                actual_rows
                == EXPECTED_ROWS[table_name]
            ),
            "Selected_Columns": actual_columns,
            "Expected_Selected_Columns": len(
                SELECTED_COLUMNS[table_name]
            ),
            "Columns_Match": (
                actual_columns
                == len(SELECTED_COLUMNS[table_name])
            ),
            "Memory_MB": round(memory_mb, 2),
        }
    )


shape_check_df = pd.DataFrame(shape_results)

display(shape_check_df)

,Table,Expected_Rows,Actual_Rows,Rows_Match,Selected_Columns,Expected_Selected_Columns,Columns_Match,Memory_MB
0,Collisions,513801,513801,True,17,17,True,99.99
1,Vehicles,937265,937265,True,8,8,True,68.83
2,Casualties,652821,652821,True,8,8,True,52.92


In [40]:
# Cell 8 Xem thử dữ liệu và kiểu dữ liệu
for table_name, dataframe in DATAFRAMES.items():
    print(f"\n===== {table_name.upper()} =====")

    display(dataframe.head(3))

    type_summary = pd.DataFrame(
        {
            "Column": dataframe.columns,
            "DataType": [
                str(dataframe[column].dtype)
                for column in dataframe.columns
            ],
        }
    )

    display(type_summary)


===== COLLISIONS =====


,collision_index,longitude,latitude,collision_severity,date,time,local_authority_ons_district,first_road_class,road_type,speed_limit,junction_detail,junction_control,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,urban_or_rural_area
0,202517M102225,-1.23119,54.56929,3,15/02/2025,19:15,E06000002,6,6,30,16,4,0,4,1,1,1
1,202417S111924,-1.31046,54.57776,2,22/10/2024,14:48,E06000004,3,6,30,0,-1,0,1,1,1,1
2,2025111687011,-1.57237,54.53082,3,19/12/2025,06:40,E06000005,6,6,30,13,4,0,4,2,2,1


,Column,DataType
0,collision_index,string
1,longitude,float64
2,latitude,float64
3,collision_severity,category
4,date,string
5,time,string
6,local_authority_ons_district,category
7,first_road_class,category
8,road_type,category
9,speed_limit,category



===== VEHICLES =====


,collision_index,vehicle_reference,vehicle_type,vehicle_manoeuvre,skidding_and_overturning,vehicle_leaving_carriageway,sex_of_driver,age_band_of_driver
0,2021461109918,1,9,19,0,0,1,8
1,202131B041421,1,9,19,0,0,2,9
2,2021010310120,1,9,5,0,0,1,6


,Column,DataType
0,collision_index,string
1,vehicle_reference,Int64
2,vehicle_type,category
3,vehicle_manoeuvre,category
4,skidding_and_overturning,category
5,vehicle_leaving_carriageway,category
6,sex_of_driver,category
7,age_band_of_driver,category



===== CASUALTIES =====


,collision_index,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_band_of_casualty,casualty_severity,casualty_type
0,2021010287661,1,1,3,1,8,3,0
1,2021010293865,1,1,3,1,6,3,0
2,2021010298600,1,1,3,1,2,3,0


,Column,DataType
0,collision_index,string
1,vehicle_reference,Int64
2,casualty_reference,Int64
3,casualty_class,category
4,sex_of_casualty,category
5,age_band_of_casualty,category
6,casualty_severity,category
7,casualty_type,category


In [41]:
# Cell 9 Kiểm tra duplicate theo natural key
duplicate_results = []

for table_name, dataframe in DATAFRAMES.items():

    natural_key = NATURAL_KEYS[table_name]

    # Số dòng nằm trong các nhóm khóa bị trùng
    duplicated_group_rows = int(
        dataframe.duplicated(
            subset=natural_key,
            keep=False,
        ).sum()
    )

    # Số dòng dư nếu giữ lại bản ghi đầu tiên
    excess_duplicate_rows = int(
        dataframe.duplicated(
            subset=natural_key,
            keep="first",
        ).sum()
    )

    # Số tổ hợp khóa duy nhất
    distinct_keys = (
        dataframe[natural_key]
        .drop_duplicates()
        .shape[0]
    )

    duplicate_results.append(
        {
            "Table": table_name,
            "Natural_Key": " + ".join(natural_key),
            "Total_Rows": len(dataframe),
            "Distinct_Keys": distinct_keys,
            "Rows_In_Duplicate_Groups": duplicated_group_rows,
            "Excess_Duplicate_Rows": excess_duplicate_rows,
            "Key_Is_Unique": excess_duplicate_rows == 0,
        }
    )


duplicate_check_df = pd.DataFrame(
    duplicate_results
)

display(duplicate_check_df)

,Table,Natural_Key,Total_Rows,Distinct_Keys,Rows_In_Duplicate_Groups,Excess_Duplicate_Rows,Key_Is_Unique
0,Collisions,collision_index,513801,513801,0,0,True
1,Vehicles,collision_index + vehicle_reference,937265,937265,0,0,True
2,Casualties,collision_index + casualty_reference,652821,652821,0,0,True


In [42]:
# Cell 10 Kiểm tra quan hệ và orphan
# ============================================================
# 1. Collision được tham chiếu bởi Vehicles và Casualties
# ============================================================

collision_ids = set(
    df_collisions["collision_index"]
    .dropna()
    .unique()
)


vehicle_orphan_collision_mask = (
    ~df_vehicles["collision_index"].isin(
        collision_ids
    )
)


casualty_orphan_collision_mask = (
    ~df_casualties["collision_index"].isin(
        collision_ids
    )
)


# ============================================================
# 2. Casualty có tìm thấy đúng Vehicle không
# ============================================================

vehicle_pairs = (
    df_vehicles[
        [
            "collision_index",
            "vehicle_reference",
        ]
    ]
    .dropna()
    .drop_duplicates()
)


casualty_vehicle_pairs = (
    df_casualties[
        [
            "collision_index",
            "vehicle_reference",
        ]
    ]
    .dropna()
    .drop_duplicates()
)


casualty_to_vehicle_check = (
    casualty_vehicle_pairs.merge(
        vehicle_pairs,
        on=[
            "collision_index",
            "vehicle_reference",
        ],
        how="left",
        indicator=True,
    )
)


orphan_casualty_vehicle_pairs = int(
    (
        casualty_to_vehicle_check["_merge"]
        == "left_only"
    ).sum()
)


# ============================================================
# 3. Vehicle nào không được casualty tham chiếu
# ============================================================

vehicle_reference_check = (
    vehicle_pairs.merge(
        casualty_vehicle_pairs,
        on=[
            "collision_index",
            "vehicle_reference",
        ],
        how="left",
        indicator=True,
    )
)


vehicles_without_casualty = int(
    (
        vehicle_reference_check["_merge"]
        == "left_only"
    ).sum()
)


relationship_check_df = pd.DataFrame(
    [
        {
            "Check": (
                "Vehicle tham chiếu collision "
                "không tồn tại"
            ),
            "Problem_Count": int(
                vehicle_orphan_collision_mask.sum()
            ),
        },
        {
            "Check": (
                "Casualty tham chiếu collision "
                "không tồn tại"
            ),
            "Problem_Count": int(
                casualty_orphan_collision_mask.sum()
            ),
        },
        {
            "Check": (
                "Cặp collision + vehicle của "
                "casualty không tồn tại trong Vehicles"
            ),
            "Problem_Count": (
                orphan_casualty_vehicle_pairs
            ),
        },
        {
            "Check": (
                "Vehicle không được casualty "
                "nào tham chiếu"
            ),
            "Problem_Count": (
                vehicles_without_casualty
            ),
        },
    ]
)


display(relationship_check_df)

,Check,Problem_Count
0,Vehicle tham chiếu collision không tồn tại,0
1,Casualty tham chiếu collision không tồn tại,0
2,Cặp collision + vehicle của casualty không tồn...,0
3,Vehicle không được casualty nào tham chiếu,370320


In [43]:
# Cell 11 Tổng hợp NULL và giá trị rỗng
missing_results = []


for table_name, dataframe in DATAFRAMES.items():

    for column in dataframe.columns:

        series = dataframe[column]

        null_mask = series.isna()

        # Chỉ kiểm tra chuỗi rỗng cho kiểu string/category
        if (
            str(series.dtype) == "string"
            or str(series.dtype) == "category"
        ):
            blank_mask = (
                series.astype("string")
                .str.strip()
                .eq("")
                .fillna(False)
            )
        else:
            blank_mask = pd.Series(
                False,
                index=series.index,
            )

        missing_mask = null_mask | blank_mask

        missing_count = int(
            missing_mask.sum()
        )

        distinct_non_missing = int(
            series.loc[~missing_mask]
            .nunique(dropna=True)
        )

        missing_results.append(
            {
                "Table": table_name,
                "Column": column,
                "DataType": str(series.dtype),
                "Total_Rows": len(series),
                "Null_Count": int(
                    null_mask.sum()
                ),
                "Blank_Count": int(
                    blank_mask.sum()
                ),
                "Missing_Total": missing_count,
                "Missing_Percent": round(
                    missing_count
                    / len(series)
                    * 100,
                    4,
                ),
                "Distinct_Non_Missing": (
                    distinct_non_missing
                ),
            }
        )


missing_summary_df = pd.DataFrame(
    missing_results
)


display(
    missing_summary_df.sort_values(
        by=[
            "Missing_Percent",
            "Table",
            "Column",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
)

,Table,Column,DataType,Total_Rows,Null_Count,Blank_Count,Missing_Total,Missing_Percent,Distinct_Non_Missing
2,Collisions,latitude,float64,513801,53,0,53,0.0103,412450
1,Collisions,longitude,float64,513801,53,0,53,0.0103,437862
30,Casualties,age_band_of_casualty,category,652821,0,0,0,0.0000,12
28,Casualties,casualty_class,category,652821,0,0,0,0.0000,3
27,Casualties,casualty_reference,Int64,652821,0,0,0,0.0000,145
31,Casualties,casualty_severity,category,652821,0,0,0,0.0000,3
32,Casualties,casualty_type,category,652821,0,0,0,0.0000,23
25,Casualties,collision_index,string,652821,0,0,0,0.0000,513801
29,Casualties,sex_of_casualty,category,652821,0,0,0,0.0000,4
26,Casualties,vehicle_reference,Int64,652821,0,0,0,0.0000,38


không phải NULL, nên chưa được tính là missing. \
Cell 11 chỉ kiểm tra:
- NULL thật.
- Giá trị trống.
- Số lượng giá trị khác nhau.
Sau bước này, ta mới kiểm tra từng mã phân loại để phân biệt:
- Giá trị hợp lệ.
- Unknown.
- Not reported.
- Not applicable.
- Mã không có trong Data Guide.

Kết luận tạm thời về tọa độ \
53 collision thiếu tọa độ chỉ ảnh hưởng đến báo cáo bản đồ. Không nên loại cả collision hoặc casualty khỏi kho dữ liệu. \
Quy tắc dự kiến:
- Giữ longitude và latitude là nullable.
- Các bản ghi thiếu tọa độ vẫn được nạp.
- Khi làm bản đồ Power BI, loại 53 collision thiếu tọa độ khỏi riêng biểu đồ bản đồ.
- Không tự điền tọa độ trung bình hoặc 0, vì sẽ tạo vị trí giả.
Trước khi chốt, cần kiểm tra hai tọa độ có cùng thiếu trên đúng 53 dòng hay không.

In [44]:
# Cell 12 Kiểm tra cặp tọa độ
longitude_missing = (
    df_collisions["longitude"].isna()
)

latitude_missing = (
    df_collisions["latitude"].isna()
)


coordinate_check_df = pd.DataFrame(
    [
        {
            "Check": "Thiếu cả longitude và latitude",
            "Count": int(
                (
                    longitude_missing
                    & latitude_missing
                ).sum()
            ),
        },
        {
            "Check": "Chỉ thiếu longitude",
            "Count": int(
                (
                    longitude_missing
                    & ~latitude_missing
                ).sum()
            ),
        },
        {
            "Check": "Chỉ thiếu latitude",
            "Count": int(
                (
                    ~longitude_missing
                    & latitude_missing
                ).sum()
            ),
        },
        {
            "Check": "Có đầy đủ cả hai tọa độ",
            "Count": int(
                (
                    ~longitude_missing
                    & ~latitude_missing
                ).sum()
            ),
        },
    ]
)


display(coordinate_check_df)


,Check,Count
0,Thiếu cả longitude và latitude,53
1,Chỉ thiếu longitude,0
2,Chỉ thiếu latitude,0
3,Có đầy đủ cả hai tọa độ,513748


In [45]:
coordinate_range_df = pd.DataFrame(
    [
        {
            "Column": "longitude",
            "Minimum": df_collisions["longitude"].min(),
            "Maximum": df_collisions["longitude"].max(),
            "Invalid_Global_Range": int(
                (
                    (
                        df_collisions["longitude"] < -180
                    )
                    | (
                        df_collisions["longitude"] > 180
                    )
                ).sum()
            ),
        },
        {
            "Column": "latitude",
            "Minimum": df_collisions["latitude"].min(),
            "Maximum": df_collisions["latitude"].max(),
            "Invalid_Global_Range": int(
                (
                    (
                        df_collisions["latitude"] < -90
                    )
                    | (
                        df_collisions["latitude"] > 90
                    )
                ).sum()
            ),
        },
    ]
)


display(coordinate_range_df)

,Column,Minimum,Maximum,Invalid_Global_Range
0,longitude,-7.486852,1.759829,0
1,latitude,49.912210,60.500008,0


In [46]:
# Cell 13 Kiểm tra ngày
parsed_collision_date = pd.to_datetime(
    df_collisions["date"],
    errors="coerce",
    dayfirst=True,
)


date_check_df = pd.DataFrame(
    [
        {
            "Check": "Tổng số dòng",
            "Value": len(
                df_collisions
            ),
        },
        {
            "Check": "Ngày không chuyển đổi được",
            "Value": int(
                parsed_collision_date
                .isna()
                .sum()
            ),
        },
        {
            "Check": "Ngày nhỏ nhất",
            "Value": (
                parsed_collision_date.min()
            ),
        },
        {
            "Check": "Ngày lớn nhất",
            "Value": (
                parsed_collision_date.max()
            ),
        },
        {
            "Check": "Số ngày khác nhau",
            "Value": int(
                parsed_collision_date.nunique()
            ),
        },
    ]
)


display(date_check_df)

,Check,Value
0,Tổng số dòng,513801
1,Ngày không chuyển đổi được,0
2,Ngày nhỏ nhất,2021-01-01 00:00:00
3,Ngày lớn nhất,2025-12-31 00:00:00
4,Số ngày khác nhau,1826


In [47]:
# Phân bố collision theo năm
collision_year_distribution_df = (
    parsed_collision_date
    .dt.year
    .value_counts()
    .sort_index()
    .rename_axis("Year")
    .reset_index(name="Collision_Count")
)


collision_year_distribution_df[
    "Percent"
] = (
    collision_year_distribution_df[
        "Collision_Count"
    ]
    / len(df_collisions)
    * 100
).round(2)


display(
    collision_year_distribution_df
)

,Year,Collision_Count,Percent
0,2021,101087,19.67
1,2022,106004,20.63
2,2023,104258,20.29
3,2024,100927,19.64
4,2025,101525,19.76


In [48]:
# Kiểm tra năm ngoài phạm vi
expected_years = {
    2021,
    2022,
    2023,
    2024,
    2025,
}


actual_years = set(
    parsed_collision_date
    .dropna()
    .dt.year
    .unique()
)


unexpected_years = (
    actual_years
    - expected_years
)


print(
    "Các năm thực tế:",
    sorted(actual_years),
)

print(
    "Các năm ngoài phạm vi:",
    sorted(unexpected_years),
)

Các năm thực tế: [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Các năm ngoài phạm vi: []


In [49]:
# Cell 14 Kiểm tra định dạng thời gian
time_text = (
    df_collisions["time"]
    .astype("string")
    .str.strip()
)


valid_time_mask = (
    time_text.str.fullmatch(
        r"(?:[01]\d|2[0-3]):[0-5]\d"
    )
    .fillna(False)
)


time_check_df = pd.DataFrame(
    [
        {
            "Check": "Tổng số dòng",
            "Value": len(
                df_collisions
            ),
        },
        {
            "Check": "Thời gian hợp lệ HH:MM",
            "Value": int(
                valid_time_mask.sum()
            ),
        },
        {
            "Check": "Thời gian không hợp lệ",
            "Value": int(
                (~valid_time_mask).sum()
            ),
        },
        {
            "Check": "Số thời điểm khác nhau",
            "Value": int(
                time_text.nunique()
            ),
        },
        {
            "Check": "Thời gian nhỏ nhất",
            "Value": time_text.min(),
        },
        {
            "Check": "Thời gian lớn nhất",
            "Value": time_text.max(),
        },
    ]
)


display(time_check_df)

,Check,Value
0,Tổng số dòng,513801
1,Thời gian hợp lệ HH:MM,513801
2,Thời gian không hợp lệ,0
3,Số thời điểm khác nhau,1440
4,Thời gian nhỏ nhất,00:00
5,Thời gian lớn nhất,23:59


In [50]:
# Cell 15 Liệt kê các mã đang tồn tại
CODED_COLUMNS = {
    "Collisions": [
        "collision_severity",
        "first_road_class",
        "road_type",
        "speed_limit",
        "junction_detail",
        "junction_control",
        "pedestrian_crossing",
        "light_conditions",
        "weather_conditions",
        "road_surface_conditions",
        "urban_or_rural_area",
    ],

    "Vehicles": [
        "vehicle_type",
        "vehicle_manoeuvre",
        "age_band_of_driver",
        "sex_of_driver",
        "skidding_and_overturning",
        "vehicle_leaving_carriageway",
    ],

    "Casualties": [
        "casualty_class",
        "sex_of_casualty",
        "age_band_of_casualty",
        "casualty_severity",
        "casualty_type",
    ],
}


def smart_sort(values):
    """
    Sắp xếp mã số theo thứ tự số học.
    Nếu gặp giá trị không phải số thì xếp sau.
    """

    def sort_key(value):
        try:
            return (
                0,
                float(value),
            )
        except (TypeError, ValueError):
            return (
                1,
                str(value),
            )

    return sorted(
        values,
        key=sort_key,
    )


code_inventory_results = []


for table_name, columns in CODED_COLUMNS.items():

    dataframe = DATAFRAMES[table_name]

    for column in columns:

        unique_values = (
            dataframe[column]
            .dropna()
            .astype("string")
            .str.strip()
            .unique()
            .tolist()
        )

        sorted_values = smart_sort(
            unique_values
        )

        code_inventory_results.append(
            {
                "Table": table_name,
                "Column": column,
                "Distinct_Code_Count": len(
                    sorted_values
                ),
                "Raw_Codes": ", ".join(
                    map(
                        str,
                        sorted_values,
                    )
                ),
            }
        )


code_inventory_df = pd.DataFrame(
    code_inventory_results
)


display(code_inventory_df)

,Table,Column,Distinct_Code_Count,Raw_Codes
0,Collisions,collision_severity,3,"1, 2, 3"
1,Collisions,first_road_class,7,"-1, 1, 2, 3, 4, 5, 6"
2,Collisions,road_type,6,"1, 2, 3, 6, 7, 9"
3,Collisions,speed_limit,7,"-1, 20, 30, 40, 50, 60, 70"
4,Collisions,junction_detail,8,"-1, 0, 13, 16, 17, 18, 19, 99"
5,Collisions,junction_control,6,"-1, 1, 2, 3, 4, 9"
6,Collisions,pedestrian_crossing,10,"-1, 0, 11, 12, 13, 14, 15, 16, 17, 99"
7,Collisions,light_conditions,6,"-1, 1, 4, 5, 6, 7"
8,Collisions,weather_conditions,10,"-1, 1, 2, 3, 4, 5, 6, 7, 8, 9"
9,Collisions,road_surface_conditions,7,"-1, 1, 2, 3, 4, 5, 9"


In [51]:
# Cell 16 — Đọc Data Guide
from pathlib import Path
import pandas as pd

DATA_GUIDE_PATH = Path(
    r"D:\OLAP\Data\Reference\data_guide.csv"
)

data_guide_raw = pd.read_csv(
    DATA_GUIDE_PATH,
    sep=";",
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False
)

# Chuẩn hóa tên cột để dễ sử dụng
data_guide_raw.columns = (
    data_guide_raw.columns
    .str.strip()
    .str.lower()
)

data_guide = data_guide_raw.rename(
    columns={
        "field name": "field_name",
        "code/format": "code"
    }
)

required_guide_columns = {
    "table",
    "field_name",
    "code",
    "label",
    "note"
}

missing_guide_columns = (
    required_guide_columns - set(data_guide.columns)
)

if missing_guide_columns:
    print("❌ Data Guide thiếu các cột:", missing_guide_columns)
else:
    print("✅ Đọc Data Guide thành công")
    print("Số dòng:", len(data_guide))
    display(data_guide.head(10))

✅ Đọc Data Guide thành công
Số dòng: 1821


,table,field_name,code,label,note
0,collision,collision_index,,,unique value for each collision. The collision...
1,collision,collision_year,,,
2,collision,collision_ref_no,,,In year id used by the police to reference a c...
3,collision,location_easting_osgr,,,Null if not known
4,collision,location_northing_osgr,,,Null if not known
5,collision,longitude,,,Null if not known
6,collision,latitude,,,Null if not known
7,collision,police_force,1,Metropolitan Police,
8,collision,police_force,3,Cumbria,
9,collision,police_force,4,Lancashire,


In [52]:
# Cell 17 — Làm sạch bảng mã và kiểm tra xung đột
def normalize_code(value):
    """
    Chuẩn hóa mã để 1, 1.0 và '1' có thể đối chiếu với nhau.
    Không thay đổi những mã dạng chữ.
    """
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if text == "":
        return ""

    try:
        number = float(text)

        if number.is_integer():
            return str(int(number))

    except ValueError:
        pass

    return text


# Loại khoảng trắng thừa
for column in ["table", "field_name", "code", "label", "note"]:
    data_guide[column] = data_guide[column].str.strip()

# Chuẩn hóa tên bảng và tên field
data_guide["table"] = data_guide["table"].str.lower()
data_guide["field_name"] = data_guide["field_name"].str.lower()
data_guide["code"] = data_guide["code"].map(normalize_code)

# Chỉ lấy những dòng thực sự là bảng ánh xạ mã
lookup_mapping = data_guide[
    (data_guide["table"] != "") &
    (data_guide["field_name"] != "") &
    (data_guide["code"] != "") &
    (data_guide["label"] != "")
].copy()

# Kiểm tra: cùng một table + field + code có nhiều label hay không
mapping_conflicts = (
    lookup_mapping
    .groupby(["table", "field_name", "code"])["label"]
    .nunique()
    .reset_index(name="label_count")
    .query("label_count > 1")
)

if mapping_conflicts.empty:
    print("✅ Không có mã nào ánh xạ tới nhiều label khác nhau")
else:
    print("❌ Có xung đột trong Data Guide")
    display(mapping_conflicts)

# Giữ một dòng duy nhất cho mỗi mã
lookup_mapping = lookup_mapping.drop_duplicates(
    subset=["table", "field_name", "code"],
    keep="first"
)

print("Số ánh xạ hợp lệ:", len(lookup_mapping))

✅ Không có mã nào ánh xạ tới nhiều label khác nhau
Số ánh xạ hợp lệ: 1774


In [54]:
# Cell 18 — Đối chiếu mã thực tế với Data Guide

data_tables = {
    "Collisions": df_collisions,
    "Vehicles": df_vehicles,
    "Casualties": df_casualties
}

categorical_columns = {
    "Collisions": [
        "local_authority_ons_district",
        "collision_severity",
        "first_road_class",
        "road_type",
        "speed_limit",
        "junction_detail",
        "junction_control",
        "pedestrian_crossing",
        "light_conditions",
        "weather_conditions",
        "road_surface_conditions",
        "urban_or_rural_area"
    ],
    "Vehicles": [
        "vehicle_type",
        "vehicle_manoeuvre",
        "age_band_of_driver",
        "sex_of_driver",
        "skidding_and_overturning",
        "vehicle_leaving_carriageway"
    ],
    "Casualties": [
        "casualty_class",
        "sex_of_casualty",
        "age_band_of_casualty",
        "casualty_severity",
        "casualty_type"
    ]
}

# Tên bảng trong Data Guide dùng dạng số ít
guide_table_names = {
    "Collisions": "collision",
    "Vehicles": "vehicle",
    "Casualties": "casualty"
}

actual_code_rows = []

for table_name, columns in categorical_columns.items():
    dataframe = data_tables[table_name]
    total_rows = len(dataframe)

    for column in columns:
        normalized_values = (
            dataframe[column]
            .astype("string")
            .map(normalize_code)
        )

        counts = normalized_values.value_counts(dropna=False)

        for raw_code, count in counts.items():
            actual_code_rows.append({
                "Table": table_name,
                "Guide_Table": guide_table_names[table_name],
                "Column": column,
                "Raw_Code": raw_code,
                "Count": int(count),
                "Percent": round(count / total_rows * 100, 4)
            })

actual_codes = pd.DataFrame(actual_code_rows)

mapping_for_merge = lookup_mapping.rename(
    columns={
        "table": "Guide_Table",
        "field_name": "Column",
        "code": "Raw_Code",
        "label": "Label",
        "note": "Note"
    }
)

code_comparison = actual_codes.merge(
    mapping_for_merge[
        ["Guide_Table", "Column", "Raw_Code", "Label", "Note"]
    ],
    on=["Guide_Table", "Column", "Raw_Code"],
    how="left"
)

code_comparison["Mapping_Status"] = code_comparison["Label"].apply(
    lambda value: "MAPPED" if pd.notna(value) else "UNMAPPED"
)

code_comparison = code_comparison.sort_values(
    ["Table", "Column", "Raw_Code"]
).reset_index(drop=True)

display(code_comparison)

,Table,Guide_Table,Column,Raw_Code,Count,Percent,Label,Note,Mapping_Status
0,Casualties,casualty,age_band_of_casualty,-1,14106,2.1608,Data missing or out of range,,MAPPED
1,Casualties,casualty,age_band_of_casualty,1,10419,1.5960,0 - 5,,MAPPED
2,Casualties,casualty,age_band_of_casualty,10,33977,5.2046,66 - 75,,MAPPED
3,Casualties,casualty,age_band_of_casualty,11,28881,4.4240,Over 75,,MAPPED
4,Casualties,casualty,age_band_of_casualty,2,16597,2.5424,6 - 10,,MAPPED
...,...,...,...,...,...,...,...,...,...
560,Vehicles,vehicle,vehicle_type,9,637681,68.0364,Car,category introduced in 2005 specification,MAPPED
561,Vehicles,vehicle,vehicle_type,90,5094,0.5435,Other vehicle,cateogry introduced in 2011 specification,MAPPED
562,Vehicles,vehicle,vehicle_type,97,2988,0.3188,Motorcycle - unknown cc,cateogry introduced in 2011 specification,MAPPED
563,Vehicles,vehicle,vehicle_type,98,3920,0.4182,Goods vehicle - unknown weight,cateogry introduced in 2011 specification,MAPPED


In [55]:
# Cell 19 — Tìm những mã chưa giải nghĩa được
unmapped_codes = code_comparison[
    code_comparison["Mapping_Status"] == "UNMAPPED"
][
    ["Table", "Column", "Raw_Code", "Count", "Percent"]
]

if unmapped_codes.empty:
    print("✅ Tất cả mã thực tế đều được giải nghĩa trong Data Guide")
else:
    print(
        f"⚠️ Có {len(unmapped_codes)} mã thực tế chưa được giải nghĩa"
    )
    display(unmapped_codes)

⚠️ Có 9 mã thực tế chưa được giải nghĩa


,Table,Column,Raw_Code,Count,Percent
318,Collisions,local_authority_ons_district,E07000246,370,0.0720
355,Collisions,local_authority_ons_district,E08000038,364,0.0708
356,Collisions,local_authority_ons_district,E08000039,838,0.1631
469,Collisions,speed_limit,20,88096,17.1459
470,Collisions,speed_limit,30,267325,52.0289
471,Collisions,speed_limit,40,44750,8.7096
472,Collisions,speed_limit,50,22666,4.4114
473,Collisions,speed_limit,60,62703,12.2038
474,Collisions,speed_limit,70,28258,5.4998


In [56]:
# Cell 20 — Xem riêng các giá trị Unknown và Not Applicable
special_value_pattern = (
    r"unknown|not known|not reported|not applicable|"
    r"missing|out of range|untraced"
)

special_codes = code_comparison[
    code_comparison["Label"]
    .fillna("")
    .str.contains(
        special_value_pattern,
        case=False,
        regex=True
    )
][
    [
        "Table",
        "Column",
        "Raw_Code",
        "Label",
        "Note",
        "Count",
        "Percent"
    ]
].sort_values(
    ["Table", "Column", "Raw_Code"]
)

if special_codes.empty:
    print("Không tìm thấy nhãn Unknown/Not Applicable theo từ khóa.")
else:
    display(special_codes)

,Table,Column,Raw_Code,Label,Note,Count,Percent
0,Casualties,age_band_of_casualty,-1,Data missing or out of range,,14106,2.1608
18,Casualties,casualty_type,-1,Data missing or out of range,,5780,0.8854
38,Casualties,casualty_type,97,Motorcycle - unknown cc rider or passenger,introduced in 2011 specification,1742,0.2668
39,Casualties,casualty_type,98,Goods vehicle (unknown weight) occupant,introduced in 2011 specification,535,0.0820
40,Casualties,casualty_type,99,Unknown vehicle type (self rep only),introduced in 2011 specification,33,0.0051
41,Casualties,sex_of_casualty,-1,Data missing or out of range,,6304,0.9657
44,Casualties,sex_of_casualty,9,unknown (self reported),,129,0.0198
48,Collisions,first_road_class,-1,Data missing or out of range,,6,0.0012
55,Collisions,junction_control,-1,Data missing or out of range,,217897,42.4088
60,Collisions,junction_control,9,unknown (self reported),,11210,2.1818


In [57]:
# Cell 21 — Xem đầy đủ ghi chú thay đổi specification
pd.set_option("display.max_colwidth", None)

fields_to_investigate = [
    ("collision", "junction_detail"),
    ("collision", "pedestrian_crossing"),
    ("vehicle", "vehicle_manoeuvre")
]

note_rows = []

for table_name, field_name in fields_to_investigate:
    field_notes = (
        lookup_mapping[
            (lookup_mapping["table"] == table_name) &
            (lookup_mapping["field_name"] == field_name) &
            (lookup_mapping["note"] != "")
        ]["note"]
        .drop_duplicates()
        .tolist()
    )

    note_rows.append({
        "Table": table_name,
        "Field": field_name,
        "Notes": " | ".join(field_notes)
    })

field_notes_summary = pd.DataFrame(note_rows)

display(field_notes_summary)

,Table,Field,Notes
0,collision,junction_detail,New field added with the 2024 data - data collected in or converted to the 2024 STATS19 specification
1,collision,pedestrian_crossing,New field added with the 2024 data - data collected in or converted to the 2024 STATS19 specification
2,vehicle,vehicle_manoeuvre,New field added with the 2024 data - data collected in or converted to the 2024 STATS19 specification


In [58]:
# Cell 22 — Kiểm tra ba cột thay đổi theo từng năm
# Lấy năm từ date của bảng Collisions
collision_year_lookup = df_collisions[
    ["collision_index", "date"]
].copy()

collision_year_lookup["Year"] = pd.to_datetime(
    collision_year_lookup["date"],
    format="%d/%m/%Y",
    errors="coerce"
).dt.year

# Gắn năm cho bảng Vehicles
vehicles_with_year = df_vehicles.merge(
    collision_year_lookup[["collision_index", "Year"]],
    on="collision_index",
    how="left",
    validate="many_to_one"
)

tables_with_year = {
    "Collisions": df_collisions.merge(
        collision_year_lookup[["collision_index", "Year"]],
        on="collision_index",
        how="left",
        validate="one_to_one"
    ),
    "Vehicles": vehicles_with_year
}

fields_by_year = [
    ("Collisions", "junction_detail"),
    ("Collisions", "pedestrian_crossing"),
    ("Vehicles", "vehicle_manoeuvre")
]

for table_name, column in fields_by_year:
    temp = tables_with_year[table_name][
        ["Year", column]
    ].copy()

    temp["Raw_Code"] = (
        temp[column]
        .astype("string")
        .map(normalize_code)
    )

    profile = pd.crosstab(
        index=temp["Year"],
        columns=temp["Raw_Code"],
        normalize="index"
    ) * 100

    # Thêm label vào tiêu đề cột
    label_lookup = (
        code_comparison[
            (code_comparison["Table"] == table_name) &
            (code_comparison["Column"] == column)
        ]
        .set_index("Raw_Code")["Label"]
        .to_dict()
    )

    profile.columns = [
        f"{code}: {label_lookup.get(code, 'No label')}"
        for code in profile.columns
    ]

    print(f"\n{table_name} — {column}")
    print("Đơn vị: % số dòng trong từng năm")

    display(profile.round(2))


Collisions — junction_detail
Đơn vị: % số dòng trong từng năm


,-1: Data missing or out of range,0: Not at junction or within 20 metres,13: T or staggered junction,16: Crossroads,17: Junction with more than four arms (not roundabout),18: Using private drive or entrance,19: Other junction,99: unknown (self reported)
Year,,,,,,,,
2021,2.04,51.61,27.40,9.27,1.50,2.19,5.99,0.00
2022,1.71,52.02,27.98,9.19,1.54,2.15,5.42,0.00
2023,2.30,51.06,27.85,9.12,1.57,2.17,5.93,0.00
2024,6.94,45.72,27.75,9.41,1.65,2.24,6.30,0.00
2025,6.60,44.65,27.77,9.31,1.48,2.11,6.66,1.41



Collisions — pedestrian_crossing
Đơn vị: % số dòng trong từng năm


,-1: Data missing or out of range,0: No physical crossing facility within 50m,11: Human crossing control by school crossing patrol,12: Human crossing control by other authorised person,13: Zebra crossing,14: Pedestrian light crossing (pelican or puffin or toucan or similar),15: Pedestrian phase at traffic signal,16: Footbridge or subway,17: Central refuge - no other controls,99: unknown (self reported)
Year,,,,,,,,,,
2021,0.22,76.16,0.43,1.20,4.17,5.25,7.52,0.26,2.41,2.37
2022,0.65,75.61,0.46,1.31,4.06,5.58,7.36,0.25,2.43,2.30
2023,2.65,71.86,0.42,1.24,4.03,5.47,7.27,0.24,2.48,4.33
2024,1.02,74.10,0.37,0.99,4.08,6.22,7.27,0.24,2.72,2.98
2025,6.43,73.42,0.25,0.56,3.70,6.49,5.95,0.18,2.72,0.30



Vehicles — vehicle_manoeuvre
Đơn vị: % số dòng trong từng năm


,-1: Data missing or out of range,1: Reversing,10: Waiting to turn right,11: Changing lane to left,12: Changing lane to right,13: Over taking moving vehicle on its offside,14: Overtaking stationary vehicle on its offside,15: Overtaking on nearside (passengers side nearest kerb),19: Going ahead,2: Parked,20: Parking,3: Waiting to go ahead,4: Slowing or stopping,5: Moving off,6: U-turn,7: Turning left,8: Waiting to turn left,9: Turning right,99: unknown (self reported)
Year,,,,,,,,,,,,,,,,,,,
2021,0.72,1.31,1.25,0.77,0.80,1.89,0.97,0.70,51.90,4.34,0.00,4.11,5.75,4.57,0.69,3.26,0.45,8.50,8.01
2022,1.20,1.26,1.21,0.74,0.84,1.72,0.91,0.69,51.89,4.48,0.00,4.06,5.51,4.57,0.66,3.22,0.42,8.64,8.00
2023,1.27,1.21,1.14,0.71,0.82,1.77,0.88,0.68,51.09,4.29,0.00,4.01,5.53,4.77,0.65,3.19,0.42,8.50,9.06
2024,3.04,1.26,1.19,0.74,0.85,1.79,0.87,0.72,50.44,4.36,0.11,4.01,5.41,4.81,0.64,3.15,0.43,8.45,7.74
2025,2.83,1.21,1.20,0.76,0.88,1.85,0.84,0.68,50.77,4.43,0.21,3.97,5.21,4.79,0.66,3.28,0.47,8.86,7.11


In [59]:
# Cell 23 — Điều tra 42,41% junction_control = -1
junction_check = df_collisions[
    ["junction_detail", "junction_control"]
].copy()

junction_check["Junction_Detail_Code"] = (
    junction_check["junction_detail"]
    .astype("string")
    .map(normalize_code)
)

junction_check["Junction_Control_Code"] = (
    junction_check["junction_control"]
    .astype("string")
    .map(normalize_code)
)

# Lấy label cho junction_detail
detail_labels = (
    code_comparison[
        (code_comparison["Table"] == "Collisions") &
        (code_comparison["Column"] == "junction_detail")
    ]
    .set_index("Raw_Code")["Label"]
    .to_dict()
)

# Lấy label cho junction_control
control_labels = (
    code_comparison[
        (code_comparison["Table"] == "Collisions") &
        (code_comparison["Column"] == "junction_control")
    ]
    .set_index("Raw_Code")["Label"]
    .to_dict()
)

junction_check["Junction_Detail"] = (
    junction_check["Junction_Detail_Code"]
    .map(lambda code: f"{code}: {detail_labels.get(code, 'No label')}")
)

junction_check["Junction_Control"] = (
    junction_check["Junction_Control_Code"]
    .map(lambda code: f"{code}: {control_labels.get(code, 'No label')}")
)

junction_relationship = pd.crosstab(
    index=junction_check["Junction_Detail"],
    columns=junction_check["Junction_Control"],
    normalize="index"
) * 100

display(junction_relationship.round(2))

Junction_Control,-1: Data missing or out of range,1: Authorised person,2: Auto traffic signal,3: Stop sign,4: Give way or uncontrolled,9: unknown (self reported)
Junction_Detail,,,,,,
-1: Data missing or out of range,2.24,0.46,12.97,0.73,48.68,34.93
0: Not at junction or within 20 metres,84.90,0.10,2.18,0.08,12.61,0.13
13: T or staggered junction,0.98,0.69,11.51,1.06,84.82,0.94
16: Crossroads,0.54,0.61,46.90,2.35,48.98,0.61
17: Junction with more than four arms (not roundabout),0.93,1.43,53.37,1.13,42.81,0.33
18: Using private drive or entrance,5.97,0.66,2.14,0.31,87.87,3.03
19: Other junction,3.39,1.26,15.90,1.19,75.57,2.69
99: unknown (self reported),3.01,0.14,12.88,0.70,7.77,75.51


In [61]:
# Cell 24 — Xác nhận chính xác nguồn gốc của mã -1
junction_minus_one = df_collisions[
    df_collisions["junction_control"]
    .astype("string")
    .map(normalize_code)
    .eq("-1")
].copy()

junction_minus_one["Junction_Detail_Code"] = (
    junction_minus_one["junction_detail"]
    .astype("string")
    .map(normalize_code)
)

junction_minus_one_summary = (
    junction_minus_one["Junction_Detail_Code"]
    .value_counts(dropna=False)
    .rename_axis("Junction_Detail_Code")
    .reset_index(name="Count")
)

junction_minus_one_summary["Percent_Within_Control_Minus_1"] = (
    junction_minus_one_summary["Count"]
    / len(junction_minus_one)
    * 100
).round(4)

junction_minus_one_summary["Junction_Detail_Label"] = (
    junction_minus_one_summary["Junction_Detail_Code"]
    .map(detail_labels)
)

junction_minus_one_summary = junction_minus_one_summary[
    [
        "Junction_Detail_Code",
        "Junction_Detail_Label",
        "Count",
        "Percent_Within_Control_Minus_1"
    ]
]

print(
    "Tổng số bản ghi junction_control = -1:",
    len(junction_minus_one)
)

display(junction_minus_one_summary)

Tổng số bản ghi junction_control = -1: 217897


,Junction_Detail_Code,Junction_Detail_Label,Count,Percent_Within_Control_Minus_1
0,0,Not at junction or within 20 metres,213958,98.1923
1,13,T or staggered junction,1397,0.6411
2,19,Other junction,1053,0.4833
3,18,Using private drive or entrance,666,0.3056
4,-1,Data missing or out of range,447,0.2051
5,16,Crossroads,259,0.1189
6,17,Junction with more than four arms (not roundabout),74,0.0340
7,99,unknown (self reported),43,0.0197
